In [5]:
html_code = r'''
<!DOCTYPE html>
<html lang="ko">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>질문 카드 신고 현황</title>

<style>
* { box-sizing: border-box; }
body {
  margin: 0;
  background: #f5f6fa;
  color: #243047;
  font-family: system-ui, "Malgun Gothic", sans-serif;
}
main { max-width: 1400px; margin: auto; padding: 28px; }
h1 { font-size: 26px; margin-bottom: 8px; }
h2 { font-size: 16px; margin: 0 0 16px; }
p { line-height: 1.7; }
small, .muted { color: #6e798b; }

.notice {
  background: #efedfb;
  padding: 14px;
  border-radius: 10px;
  margin: 20px 0;
  font-size: 13px;
}
.toolbar {
  display: flex;
  gap: 12px;
  align-items: end;
  flex-wrap: wrap;
}
label { font-size: 13px; }
input, select, button {
  font: inherit;
  border: 1px solid #dce1eb;
  padding: 10px;
  border-radius: 8px;
  background: white;
  color: inherit;
}
button { cursor: pointer; }
button:disabled { opacity: .6; cursor: wait; }
.primary { background: #6357ce; color: white; border: none; }

#connection { font-size: 13px; margin: 16px 0; }
.error { color: #b3344f; }

.cards, .charts {
  display: grid;
  grid-template-columns: repeat(4, minmax(0, 1fr));
  gap: 14px;
  margin-bottom: 20px;
}
.panel {
  background: white;
  border: 1px solid #e4e8f0;
  border-radius: 12px;
  padding: 18px;
  min-width: 0;
}
.card strong { display: block; font-size: 28px; margin-top: 10px; }
.card span { color: #6e798b; font-size: 13px; }

.workspace {
  display: grid;
  grid-template-columns: minmax(280px, .9fr) minmax(350px, 1.3fr);
  gap: 16px;
}
.filters { display: flex; gap: 8px; margin-bottom: 14px; }
.filters input { width: 100%; min-width: 0; }
.filters select { font-size: 12px; }

#questions { max-height: 650px; overflow: auto; }
.question {
  display: block;
  width: 100%;
  text-align: left;
  margin-bottom: 8px;
  padding: 14px;
}
.question.active { background: #f3f0ff; border-color: #a99ce3; }
.question p { margin: 8px 0; font-weight: 600; }
.question small { display: block; }

.quote {
  background: #f7f7fc;
  padding: 18px;
  border-radius: 10px;
  font-size: 19px;
  line-height: 1.7;
  font-weight: 600;
}
.metrics { display: flex; gap: 25px; margin: 20px 0; }
.metrics b { display: block; font-size: 24px; }
.section { border-top: 1px solid #e4e8f0; margin-top: 20px; padding-top: 20px; }
.gender { display: flex; height: 14px; overflow: hidden; border-radius: 8px; }
.gender-legend { display: flex; gap: 15px; flex-wrap: wrap; font-size: 13px; }
.empty { padding: 20px 0; color: #6e798b; font-size: 13px; }

.rank {
  display: flex;
  gap: 10px;
  width: 100%;
  text-align: left;
  border: none;
  border-bottom: 1px solid #eef0f5;
  border-radius: 0;
  padding: 11px 0;
  font-size: 12px;
}
.rank span {
  flex: 1;
  white-space: nowrap;
  overflow: hidden;
  text-overflow: ellipsis;
}
.bar { margin: 12px 0; }
.bar-label {
  display: flex;
  justify-content: space-between;
  gap: 10px;
  font-size: 12px;
  margin-bottom: 5px;
}
.bar-label span {
  overflow: hidden;
  text-overflow: ellipsis;
  white-space: nowrap;
}
.track { height: 8px; background: #efedf7; border-radius: 5px; overflow: hidden; }
.fill { height: 100%; background: #8a7bdb; }
.chart svg { width: 100%; height: 150px; }
.chart svg text { fill: #6e798b; font-size: 10px; }
table { width: 100%; border-collapse: collapse; font-size: 13px; }
td, th { padding: 10px 4px; border-bottom: 1px solid #eef0f5; text-align: left; }
td:last-child, th:last-child { text-align: right; }
.footer { text-align: center; color: #6e798b; font-size: 12px; margin-top: 24px; }

@media (max-width: 1050px) {
  .charts { grid-template-columns: 1fr 1fr; }
}
@media (max-width: 700px) {
  main { padding: 18px 12px; }
  .cards { grid-template-columns: 1fr 1fr; }
  .charts, .workspace { grid-template-columns: 1fr; }
  .filters { flex-wrap: wrap; }
  #questions { max-height: 350px; }
}
</style>
</head>

<body>
<main>
  <h1>질문 카드 신고 현황</h1>
  <p class="muted">기간별 신고 현황과 질문별 신고 내용을 확인합니다.</p>

  <div class="notice">
    <b>가상 데이터 전용</b> · BigQuery에 올린 가상 데이터를 조회합니다.
    수정 제안·LLM 호출·데이터 수정 기능은 없습니다.
  </div>

  <div class="toolbar">
    <label>
      시작일<br>
      <input type="date" id="startDate" value="2026-10-01">
    </label>
    <label>
      종료일<br>
      <input type="date" id="endDate" value="2026-10-31">
    </label>
    <button class="primary" id="load">기간 조회</button>
  </div>

  <p id="connection" role="status" aria-live="polite">연결 대기 중</p>

  <section class="cards" id="cards"></section>

  <section class="charts">
    <article class="panel chart">
      <h2>일별 신고 추이</h2>
      <div id="trend"></div>
    </article>
    <article class="panel chart">
      <h2>신고가 많은 질문 TOP 4</h2>
      <div id="ranking"></div>
    </article>
    <article class="panel chart">
      <h2>신고 사유 상위 4개</h2>
      <div id="reasons"></div>
      <small>자유서술 문구별 집계입니다.</small>
    </article>
    <article class="panel chart">
      <h2>질문별 신고 건수 분포</h2>
      <div id="distribution"></div>
    </article>
  </section>

  <section class="workspace">
    <article class="panel">
      <h2>질문 목록 <small id="listCount"></small></h2>
      <div class="filters">
        <input id="search" placeholder="질문 문구 또는 ID 검색" aria-label="질문 검색">
        <select id="sort" aria-label="정렬">
          <option value="report_count">신고 많은 순</option>
          <option value="unique_reporters">신고자 많은 순</option>
        </select>
      </div>
      <div id="questions"></div>
    </article>

    <article class="panel">
      <h2>선택한 질문의 신고 상세</h2>
      <div id="detail"></div>
    </article>
  </section>

  <p class="footer">
    상단 집계는 조회 기간 전체 기준입니다.
    신고 건수만으로 유해성이나 신고 원인을 확정하지 않습니다.
  </p>
</main>

<script>
// Cloud Run 서버 주소입니다. API 키가 아닙니다.
const API_BASE =
  "https://sns-moderation-api-748357521935.us-central1.run.app";

const $ = id => document.getElementById(id);

const esc = value => String(value).replace(
  /[&<>"']/g,
  c => ({
    "&": "&amp;",
    "<": "&lt;",
    ">": "&gt;",
    '"': "&quot;",
    "'": "&#39;"
  }[c])
);

let data = null;
let selected = null;
let requestId = 0;
let controller = null;

function clearScreen(message) {
  data = null;
  selected = null;

  $("cards").innerHTML = [
    "기간 내 신고", "전체 질문", "신고된 질문", "전체 고유 신고자"
  ].map(title => `
    <div class="panel card">
      <span>${title}</span><strong>—</strong>
    </div>
  `).join("");

  for (const id of [
    "trend", "ranking", "reasons",
    "distribution", "questions", "detail"
  ]) {
    $(id).innerHTML = `<div class="empty">${esc(message)}</div>`;
  }

  $("listCount").textContent = "";
}

function validate(result, start, end) {
  if (
    result.ok !== true ||
    result.synthetic !== true ||
    result.source !== "BigQuery" ||
    !result.summary ||
    !Array.isArray(result.questions) ||
    !Array.isArray(result.trend) ||
    result.period?.start !== start ||
    result.period?.end !== end
  ) {
    throw new Error("サーバー応答形式が一致しません。最新の 조회 코드를 확인해주세요.");
  }

  const validNumber = n => Number.isSafeInteger(n) && n >= 0;
  const ids = new Set();

  for (const q of result.questions) {
    if (
      typeof q.id !== "string" ||
      ids.has(q.id) ||
      typeof q.text !== "string" ||
      !validNumber(q.report_count) ||
      !validNumber(q.unique_reporters) ||
      q.unique_reporters > q.report_count ||
      !q.gender ||
      !["male", "female", "unknown"].every(k => validNumber(q.gender[k])) ||
      q.gender.male + q.gender.female + q.gender.unknown !== q.unique_reporters ||
      !Array.isArray(q.reasons) ||
      q.reasons.some(r => typeof r.reason !== "string" || !validNumber(r.count)) ||
      q.reasons.reduce((sum, r) => sum + r.count, 0) !== q.report_count
    ) {
      throw new Error("질문 집계 형식이나 합계가 맞지 않습니다.");
    }
    ids.add(q.id);
  }

  const days = new Set();

  for (const row of result.trend) {
    if (
      typeof row.date !== "string" ||
      !/^\d{4}-\d{2}-\d{2}$/.test(row.date) ||
      row.date < start ||
      row.date > end ||
      days.has(row.date) ||
      !validNumber(row.count)
    ) {
      throw new Error("일별 집계 형식이 맞지 않습니다.");
    }
    days.add(row.date);
  }

  const s = result.summary;

  if (
    !["total_reports", "total_questions", "reported_questions", "unique_reporters"]
      .every(k => validNumber(s[k])) ||
    s.total_questions !== result.questions.length ||
    s.total_reports !== result.questions.reduce((sum, q) => sum + q.report_count, 0) ||
    s.total_reports !== result.trend.reduce((sum, row) => sum + row.count, 0) ||
    s.reported_questions !== result.questions.filter(q => q.report_count > 0).length ||
    s.unique_reporters > s.total_reports
  ) {
    throw new Error("요약과 상세 데이터의 합계가 맞지 않습니다.");
  }
}

function renderOverview() {
  const s = data.summary;

  const cards = [
    ["기간 내 신고", s.total_reports, "건"],
    ["전체 질문", s.total_questions, "개"],
    ["신고된 질문", s.reported_questions, "개"],
    ["전체 고유 신고자", s.unique_reporters, "명"]
  ];

  $("cards").innerHTML = cards.map(([title, count, unit]) => `
    <div class="panel card">
      <span>${title}</span>
      <strong>${count.toLocaleString()}<small> ${unit}</small></strong>
    </div>
  `).join("");

  const top = [...data.questions]
    .filter(q => q.report_count > 0)
    .sort((a, b) => b.report_count - a.report_count)
    .slice(0, 4);

  $("ranking").innerHTML = top.map((q, i) => `
    <button class="rank" data-qid="${esc(q.id)}" title="${esc(q.text)}">
      <b>${i + 1}</b>
      <span>${esc(q.text)}</span>
      <b>${q.report_count}건</b>
    </button>
  `).join("") || '<div class="empty">기간 내 신고가 없습니다.</div>';

  const reasonCounts = new Map();

  data.questions.forEach(q => {
    q.reasons.forEach(r => {
      reasonCounts.set(
        r.reason,
        (reasonCounts.get(r.reason) || 0) + r.count
      );
    });
  });

  const reasons = [...reasonCounts]
    .sort((a, b) => b[1] - a[1])
    .slice(0, 4);

  const max = reasons[0]?.[1] || 1;

  $("reasons").innerHTML = reasons.map(([text, count]) => `
    <div class="bar" title="${esc(text)}">
      <div class="bar-label">
        <span>${esc(text)}</span><b>${count}</b>
      </div>
      <div class="track">
        <div class="fill" style="width:${100 * count / max}%"></div>
      </div>
    </div>
  `).join("") || '<div class="empty">기간 내 신고가 없습니다.</div>';

  const buckets = [
    ["0건", q => q.report_count === 0],
    ["1–9건", q => q.report_count >= 1 && q.report_count < 10],
    ["10–49건", q => q.report_count >= 10 && q.report_count < 50],
    ["50건 이상", q => q.report_count >= 50]
  ];

  $("distribution").innerHTML = `
    <table>
      <thead><tr><th>신고 건수</th><th>질문 수</th></tr></thead>
      <tbody>
        ${buckets.map(([name, test]) => `
          <tr>
            <td>${name}</td>
            <td>${data.questions.filter(test).length}개</td>
          </tr>
        `).join("")}
      </tbody>
    </table>
  `;

  renderTrend();
}

function renderTrend() {
  const counts = new Map(data.trend.map(row => [row.date, row.count]));
  const series = [];
  const end = new Date(data.period.end + "T00:00:00Z");

  for (
    let day = new Date(data.period.start + "T00:00:00Z");
    day <= end;
    day.setUTCDate(day.getUTCDate() + 1)
  ) {
    const key = day.toISOString().slice(0, 10);
    series.push({ date: key, count: counts.get(key) || 0 });
  }

  const actualMax = Math.max(...series.map(row => row.count));
  const max = Math.max(1, actualMax);
  const step = Math.max(1, Math.ceil(series.length / 5));

  const x = i => 30 + (
    series.length === 1 ? 114 : i * 228 / (series.length - 1)
  );
  const y = count => 110 - count / max * 85;

  const points = series
    .map((row, i) => `${x(i)},${y(row.count)}`)
    .join(" ");

  $("trend").innerHTML = `
    <svg viewBox="0 0 285 145" role="img" aria-label="일별 신고 추이">
      <line x1="25" y1="110" x2="265" y2="110" stroke="#dce1eb"/>
      <text x="0" y="113">0</text>
      <text x="0" y="25">${actualMax}</text>
      <polyline
        points="${points}"
        fill="none"
        stroke="#8170d3"
        stroke-width="2.5"
      />
      ${series.map((row, i) => `
        <circle
          cx="${x(i)}" cy="${y(row.count)}"
          r="2.5" fill="#8170d3"
        >
          <title>${row.date}: ${row.count}건</title>
        </circle>
        ${i % step === 0 ? `
          <text x="${x(i) - 13}" y="135">${row.date.slice(5)}</text>
        ` : ""}
      `).join("")}
    </svg>
  `;
}

function renderList() {
  if (!data) return;

  const term = $("search").value.trim().toLowerCase();
  const sortKey = $("sort").value;

  const questions = data.questions
    .filter(q => (q.text + q.id).toLowerCase().includes(term))
    .sort((a, b) => b[sortKey] - a[sortKey]);

  $("listCount").textContent =
    `${questions.length} / ${data.questions.length}개`;

  $("questions").innerHTML = questions.map(q => `
    <button
      class="question ${q.id === selected ? "active" : ""}"
      data-qid="${esc(q.id)}"
      aria-pressed="${q.id === selected}"
    >
      <small>${esc(q.id)}</small>
      <p>${esc(q.text)}</p>
      <small>
        신고 ${q.report_count}건 · 고유 신고자 ${q.unique_reporters}명
      </small>
    </button>
  `).join("") || '<div class="empty">검색 결과가 없습니다.</div>';
}

function selectQuestion(id) {
  const q = data?.questions.find(q => q.id === id);
  if (!q) return;

  selected = id;
  renderList();

  const count = q.unique_reporters;
  const percent = n => count ? (100 * n / count).toFixed(1) : "0.0";

  $("detail").innerHTML = `
    <small>
      ${esc(q.id)} · ${data.period.start} ~ ${data.period.end}
    </small>

    <div class="quote">${esc(q.text)}</div>

    <div class="metrics">
      <div>
        <b>${q.report_count}건</b>
        <small>기간 내 신고</small>
      </div>
      <div>
        <b>${count}명</b>
        <small>기간 내 고유 신고자</small>
      </div>
    </div>

    <div class="section">
      <h2>신고 내용 · 사유별 집계</h2>
      ${q.reasons.map(r => `
        <p>${esc(r.reason)} <b>(${r.count}건)</b></p>
      `).join("") || '<p class="muted">기간 내 신고가 없습니다.</p>'}
    </div>

    <div class="section">
      <h2>고유 신고자 성별 구성</h2>
      <div class="gender" role="img"
        aria-label="남 ${q.gender.male}명, 여 ${q.gender.female}명, 미상 ${q.gender.unknown}명">
        <span style="width:${percent(q.gender.male)}%;background:#8170d3"></span>
        <span style="width:${percent(q.gender.female)}%;background:#b4a7e5"></span>
        <span style="width:${percent(q.gender.unknown)}%;background:#dce1eb"></span>
      </div>
      <p class="gender-legend">
        <span>남 ${percent(q.gender.male)}% (${q.gender.male}명)</span>
        <span>여 ${percent(q.gender.female)}% (${q.gender.female}명)</span>
        <span>미상 ${percent(q.gender.unknown)}% (${q.gender.unknown}명)</span>
      </p>
      <small>
        ${count
          ? "전체 이용자나 노출자의 성별 구성은 아닙니다."
          : "신고자가 없어 비율을 산출할 수 없습니다. 0.0%는 표시용입니다."}
        노출 데이터가 없으므로 신고율은 계산하지 않습니다.
      </small>
    </div>
  `;
}

async function loadData() {
  const start = $("startDate").value;
  const end = $("endDate").value;

  if (
    !start || !end ||
    !Number.isFinite(Date.parse(start)) ||
    !Number.isFinite(Date.parse(end)) ||
    start > end ||
    (Date.parse(end) - Date.parse(start)) / 86400000 > 365
  ) {
    $("connection").className = "error";
    $("connection").textContent = "유효한 기간을 선택해주세요. 최대 366일입니다.";
    return;
  }

  const id = ++requestId;

  controller?.abort();
  controller = new AbortController();

  const ownController = controller;
  const timer = setTimeout(() => ownController.abort(), 65000);

  clearScreen("데이터를 불러오는 중입니다.");
  $("load").disabled = true;
  $("connection").className = "";
  $("connection").textContent = "Cloud Run을 통해 BigQuery 조회 중…";

  try {
    const url = new URL(API_BASE + "/api/dashboard");
    url.searchParams.set("start", start);
    url.searchParams.set("end", end);

    const response = await fetch(url, {
      signal: ownController.signal,
      credentials: "omit"
    });

    let result;

    try {
      result = await response.json();
    } catch {
      throw new Error("JSON 응답이 아닙니다. 서버 배포 상태를 확인해주세요.");
    }

    if (id !== requestId) return;

    if (!response.ok || result.ok !== true) {
      throw new Error(result.message || `조회 실패 (${response.status})`);
    }

    validate(result, start, end);

    data = result;
    renderOverview();
    renderList();

    if (data.questions.length) {
      selectQuestion(data.questions[0].id);
    } else {
      $("detail").innerHTML = '<div class="empty">질문이 없습니다.</div>';
    }

    $("connection").textContent =
      `BigQuery 연결 완료 · ${start} ~ ${end} · ` +
      `${data.summary.total_reports.toLocaleString()}건` +
      (data.cached ? " · 서버 캐시 재사용" : "");

  } catch (error) {
    if (id !== requestId) return;

    clearScreen("조회하지 못했습니다.");
    $("connection").className = "error";
    $("connection").textContent = error.name === "AbortError"
      ? "요청 시간이 초과되었습니다. 다시 조회해주세요."
      : error.message;

  } finally {
    clearTimeout(timer);
    if (id === requestId) $("load").disabled = false;
  }
}

$("load").addEventListener("click", loadData);
$("search").addEventListener("input", renderList);
$("sort").addEventListener("change", renderList);

for (const element of ["questions", "ranking"]) {
  $(element).addEventListener("click", event => {
    const button = event.target.closest("[data-qid]");
    if (!button) return;

    if (element === "ranking") $("search").value = "";

    selectQuestion(button.dataset.qid);

    if (element === "ranking") {
      $("detail").scrollIntoView({ behavior: "smooth", block: "start" });
    }
  });
}

// 페이지를 처음 열 때 자동 조회합니다.
loadData();
</script>
</body>
</html>
'''

In [6]:
from pathlib import Path

output_path = Path("index.html")
output_path.write_text(html_code, encoding="utf-8")

print(f"저장 위치: {output_path.resolve()}")

저장 위치: C:\Users\USER\Desktop\final_prj\codeit-sns-moderation\index.html
